# Sentiment DLinear+NODE — Train / Test Split

Cấu trúc thư mục dữ liệu:
```
DATASET/
  TRAIN/
    PRICE/        ← dữ liệu giá để TRAIN model
    SENTIMENT/    ← dữ liệu sentiment để TRAIN model
  TEST/
    PRICE/        ← dữ liệu giá để ĐÁNH GIÁ + DỰ BÁO 5 ngày
    SENTIMENT/    ← dữ liệu sentiment để ĐÁNH GIÁ + DỰ BÁO 5 ngày
```

**Luồng xử lý:**
1. **Cell 1** – Import & cấu hình đường dẫn
2. **Cell 2** – Định nghĩa Model
3. **Cell 3** – DataProcessor (load, merge, feature, sequence)
4. **Cell 4** – Hàm metrics & visualize
5. **Cell 5** – **TRAIN** từ `DATASET/TRAIN/PRICE` + `DATASET/TRAIN/SENTIMENT`, lưu model `.pt`
6. **Cell 6** – **TEST / INFERENCE**: load dữ liệu TEST, gọi model đã train, đánh giá & xuất dự báo 5 ngày tương lai

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import os
import glob
import json
import warnings
import random
import matplotlib.dates as mdates
from IPython.display import display

# ─────────────────────────────────────────────
# Hàm metrics bổ trợ
# ─────────────────────────────────────────────
def calculate_mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100

def calculate_da(y_true, y_pred):
    true_diff = np.diff(y_true)
    pred_diff = np.diff(y_pred)
    return np.mean((true_diff * pred_diff) > 0) * 100

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# ─────────────────────────────────────────────
# Random Seed
# ─────────────────────────────────────────────
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# ─────────────────────────────────────────────
# HYPERPARAMETERS
# ─────────────────────────────────────────────
SEQ_LEN    = 30
HIDDEN_DIM = 128
DROPOUT    = 0.2
EPOCHS     = 150
BATCH_SIZE = 32
LEARNING_RATE = 0.001

# Hyperparameters riêng cho Sentiment model
SENTIMENT_EPOCHS       = 200
SENTIMENT_HIDDEN_DIM   = 256
SENTIMENT_DROPOUT      = 0.1
SENTIMENT_LEARNING_RATE = 0.0005

FUTURE_DAYS = 5   # số ngày tương lai cần dự báo

# ─────────────────────────────────────────────
# MODEL TYPES: Train/Test cả 3 mô hình
# ─────────────────────────────────────────────
MODEL_TYPES = ['dlinear', 'node', 'hybrid']  # Chọn model type(s) để train

# ─────────────────────────────────────────────
# ĐÂY LÀ PHẦN BẠN CẦN CHỈNH — 4 ĐƯỜNG DẪN DATASET
# ─────────────────────────────────────────────
BASE_DIR = os.path.join("..", "..", "DATASET")   # <-- thay đổi nếu cần

TRAIN_PRICE_DIR     = os.path.join(BASE_DIR, "TRAIN", "PRICE")
TRAIN_SENTIMENT_DIR = os.path.join(BASE_DIR, "TRAIN", "SENTIMENT")
TEST_PRICE_DIR      = os.path.join(BASE_DIR, "TEST",  "PRICE")
TEST_SENTIMENT_DIR  = os.path.join(BASE_DIR, "TEST",  "SENTIMENT")

# Thư mục lưu chart & log
CHART_DIR = os.path.join("..", "..", "CHART", "DLINEAR+NODE")
LOG_DIR   = os.path.join("..", "..", "LOGS",  "DLINEAR+NODE")
os.makedirs(CHART_DIR, exist_ok=True)
os.makedirs(LOG_DIR,   exist_ok=True)

print("\n=== CẤU HÌNH ĐƯỜNG DẪN ===")
print(f"  TRAIN PRICE     : {os.path.abspath(TRAIN_PRICE_DIR)}")
print(f"  TRAIN SENTIMENT : {os.path.abspath(TRAIN_SENTIMENT_DIR)}")
print(f"  TEST  PRICE     : {os.path.abspath(TEST_PRICE_DIR)}")
print(f"  TEST  SENTIMENT : {os.path.abspath(TEST_SENTIMENT_DIR)}")
print(f"  CHART DIR       : {os.path.abspath(CHART_DIR)}")
print(f"  LOG   DIR       : {os.path.abspath(LOG_DIR)}")
print(f"\n[INFO] Sentiment Model: EPOCHS={SENTIMENT_EPOCHS}, HIDDEN_DIM={SENTIMENT_HIDDEN_DIM}, "
      f"DROPOUT={SENTIMENT_DROPOUT}, LR={SENTIMENT_LEARNING_RATE}")
print(f"[INFO] Model Types to train: {MODEL_TYPES}")

Using device: cpu

=== CẤU HÌNH ĐƯỜNG DẪN ===
  TRAIN PRICE     : d:\NghienCuu\NCT3\DATASET\TRAIN\PRICE
  TRAIN SENTIMENT : d:\NghienCuu\NCT3\DATASET\TRAIN\SENTIMENT
  TEST  PRICE     : d:\NghienCuu\NCT3\DATASET\TEST\PRICE
  TEST  SENTIMENT : d:\NghienCuu\NCT3\DATASET\TEST\SENTIMENT
  CHART DIR       : d:\NghienCuu\NCT3\CHART\DLINEAR+NODE
  LOG   DIR       : d:\NghienCuu\NCT3\LOGS\DLINEAR+NODE

[INFO] Sentiment Model: EPOCHS=200, HIDDEN_DIM=256, DROPOUT=0.1, LR=0.0005
[INFO] Model Types to train: ['dlinear', 'node', 'hybrid']


## Model Architecture

In [2]:
class SentimentDLinearNodeModel(nn.Module):
    def __init__(self, seq_len=30, price_dim=5, sentiment_dim=10, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.seq_len      = seq_len
        self.price_dim    = price_dim
        self.sentiment_dim = sentiment_dim

        # DLinear cho Price
        self.price_decomp = nn.Sequential(
            nn.Linear(seq_len, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, seq_len)
        )
        # DLinear cho Sentiment
        self.sentiment_decomp = nn.Sequential(
            nn.Linear(seq_len, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, seq_len)
        )
        # NODE (Neural ODE Approximation)
        self.node_layers = nn.Sequential(
            nn.Linear(price_dim + sentiment_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, price_dim + sentiment_dim)
        )
        # Predictor
        self.predictor = nn.Sequential(
            nn.Linear((price_dim + sentiment_dim) * seq_len, hidden_dim),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, price_x, sentiment_x):
        batch_size, seq_len, _ = price_x.shape
        combined_input   = torch.cat([price_x, sentiment_x], dim=-1)
        price_trend      = self.price_decomp(price_x.transpose(1, 2)).transpose(1, 2)
        sentiment_trend  = self.sentiment_decomp(sentiment_x.transpose(1, 2)).transpose(1, 2)
        combined_trend   = torch.cat([price_trend, sentiment_trend], dim=-1)
        node_output      = combined_input + self.node_layers(combined_input)
        final_features   = combined_trend + node_output
        return self.predictor(final_features.reshape(batch_size, -1))

print("Model SentimentDLinearNodeModel đã được định nghĩa.")

# ─────────────────────────────────────────────
# MODEL 2: DLinear Only (Price Only)
# ─────────────────────────────────────────────
class SentimentDLinearModel(nn.Module):
    """DLinear Model: Chỉ sử dụng Price (KHÔNG dùng Sentiment)"""
    def __init__(self, seq_len=30, price_dim=5, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.seq_len   = seq_len
        self.price_dim = price_dim

        # DLinear cho Price
        self.price_decomp = nn.Sequential(
            nn.Linear(seq_len, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, seq_len)
        )
        # Predictor
        self.predictor = nn.Sequential(
            nn.Linear(price_dim * seq_len, hidden_dim),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, price_x, sentiment_x=None):
        batch_size, seq_len, _ = price_x.shape
        price_trend = self.price_decomp(price_x.transpose(1, 2)).transpose(1, 2)
        return self.predictor(price_trend.reshape(batch_size, -1))

print("Model SentimentDLinearModel đã được định nghĩa.")

# ─────────────────────────────────────────────
# MODEL 3: NODE Only (Price Only)
# ─────────────────────────────────────────────
class SentimentNODEModel(nn.Module):
    """NODE Model: Chỉ sử dụng Price (KHÔNG dùng Sentiment)"""
    def __init__(self, seq_len=30, price_dim=5, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.seq_len   = seq_len
        self.price_dim = price_dim

        # NODE (Neural ODE Approximation)
        self.node_layers = nn.Sequential(
            nn.Linear(price_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, price_dim)
        )
        # Predictor
        self.predictor = nn.Sequential(
            nn.Linear(price_dim * seq_len, hidden_dim),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, price_x, sentiment_x=None):
        batch_size, seq_len, _ = price_x.shape
        node_output = price_x + self.node_layers(price_x)
        return self.predictor(node_output.reshape(batch_size, -1))

print("Model SentimentNODEModel đã được định nghĩa.")

# ─────────────────────────────────────────────
# Factory Function: Tạo model theo loại
# ─────────────────────────────────────────────
def get_model_by_type(model_type, seq_len=30, price_dim=5, sentiment_dim=10, hidden_dim=128, dropout=0.2):
    """Factory function: Tạo model theo loại"""
    if model_type == 'dlinear':
        return SentimentDLinearModel(seq_len=seq_len, price_dim=price_dim, hidden_dim=hidden_dim, dropout=dropout)
    elif model_type == 'node':
        return SentimentNODEModel(seq_len=seq_len, price_dim=price_dim, hidden_dim=hidden_dim, dropout=dropout)
    elif model_type == 'hybrid':
        return SentimentDLinearNodeModel(seq_len=seq_len, price_dim=price_dim, sentiment_dim=sentiment_dim, hidden_dim=hidden_dim, dropout=dropout)
    else:
        raise ValueError(f"Unknown model type: {model_type}")

print("Factory function get_model_by_type() đã sẵn sàng.")

Model SentimentDLinearNodeModel đã được định nghĩa.
Model SentimentDLinearModel đã được định nghĩa.
Model SentimentNODEModel đã được định nghĩa.
Factory function get_model_by_type() đã sẵn sàng.


## DataProcessor

In [3]:
class DataProcessor:
    """
    Load, merge, process price + sentiment data, and create sequences for training.
    """
    def __init__(self):
        self.price_scaler    = StandardScaler()
        self.sentiment_scaler = StandardScaler()
        self.target_scaler   = StandardScaler()

    def load_and_merge_data(self, price_path, sentiment_path):
        """
        Load price CSV + sentiment CSV, merge on Date.
        
        Args:
            price_path: path to price CSV
            sentiment_path: path to sentiment CSV
        
        Returns:
            merged dataframe
        """
        # Load price data
        df_price = pd.read_csv(price_path)
        if 'Date' in df_price.columns:
            df_price['Date'] = pd.to_datetime(df_price['Date'])
        
        # Load sentiment data
        df_sentiment = pd.read_csv(sentiment_path)
        if 'Date' in df_sentiment.columns:
            df_sentiment['Date'] = pd.to_datetime(df_sentiment['Date'])
        
        # Merge on Date
        df_merged = df_price.merge(df_sentiment, on='Date', how='inner', suffixes=('_price', '_sentiment'))
        
        return df_merged

    def create_features(self, df, specific_sent_cols=None):
        """
        Extract price and sentiment features.
        
        Args:
            df: merged dataframe
            specific_sent_cols: list of specific sentiment column names to use
                               (from TRAIN scaler, to ensure TRAIN/TEST consistency)
        
        Returns:
            price_feats, sent_feats, targets
        """
        # ── PRICE FEATURES ───────────────────────────────────────────────
        price_cols = [c for c in ['Lần cuối', 'Mở', 'Cao', 'Thấp',
                                   'Close', 'Open', 'High', 'Low'] if c in df.columns]
        price_feats = df[price_cols].copy()
        for col in price_cols:
            if price_feats[col].dtype == 'object':
                price_feats[col] = price_feats[col].str.replace(',', '').astype(float)
        price_feats = price_feats.ffill().bfill().fillna(0)

        # ── SENTIMENT FEATURES ───────────────────────────────────────────
        # Priority 1: Use provided column list (from TRAIN scaler)
        if specific_sent_cols is not None and len(specific_sent_cols) > 0:
            # Check if specified columns actually exist in df
            sent_cols_to_use = [c for c in specific_sent_cols if c in df.columns]
            
            # If none of the specified columns exist (they might be placeholders),
            # fall back to dynamic discovery
            if not sent_cols_to_use and specific_sent_cols[0] == 'placeholder_sentiment':
                print(f"    [INFO] Placeholder found but actual sentiment cols available → switching to dynamic discovery")
                sent_cols_to_use = []  # Will trigger fallback
            elif not sent_cols_to_use:
                print(f"    [WARNING] None of specified sentiment cols found: {specific_sent_cols}")
                sent_cols_to_use = []  # Will trigger fallback
        else:
            sent_cols_to_use = []
        
        # If Priority 1 didn't yield results, try Priorities 2 & 3
        if not sent_cols_to_use:
            # Priority 2: Use the 9 standard sentiment columns from calcu_sentiment.py
            standard_sent_cols = [
                'sentiment_score', 'impact_score', 'relevance_score', 'confidence',
                'short_term_score', 'medium_term_score', 'sentiment_momentum',
                'sentiment_volatility', 'sentiment_trend'
            ]
            sent_cols_to_use = [c for c in standard_sent_cols if c in df.columns]
            
            # Priority 3: Fallback to dynamic keyword matching if not 9 standard cols
            if len(sent_cols_to_use) < 9:
                sent_cols_to_use = [c for c in df.columns if any(
                    word in c.lower() for word in 
                    ['score', 'impact', 'relevance', 'sentiment', 'confidence',
                     'momentum', 'volatility', 'prob'])]

        if not sent_cols_to_use:
            print(f"    [WARNING] No sentiment columns found!")
            sent_feats = pd.DataFrame(0.0, index=df.index, columns=['placeholder_sentiment'])
        else:
            sent_feats = df[sent_cols_to_use].copy().fillna(0)

        target_col = next((c for c in ['Lần cuối', 'Close'] if c in df.columns), price_cols[0])
        return price_feats, sent_feats, df[target_col].values

    def prepare_sequences(self, price_feats, sentiment_feats, targets, ticker, seq_len, fit=False):
        """
        Transform features into sequences for RNN training.
        
        Args:
            price_feats: DataFrame with price features
            sentiment_feats: DataFrame with sentiment features
            targets: 1D array of target values (price to predict)
            ticker: stock name (for logging)
            seq_len: sequence length (window size)
            fit: if True, fit scalers on this data (TRAIN set); 
                 if False, use existing fitted scalers (TEST set)
        
        Returns:
            X_price, X_sentiment, y (3D tensors)
        """
        # Fit or transform scalers
        if fit:
            price_scaled = self.price_scaler.fit_transform(price_feats)
            sentiment_scaled = self.sentiment_scaler.fit_transform(sentiment_feats)
            targets_scaled = self.target_scaler.fit_transform(targets.reshape(-1, 1)).flatten()
        else:
            price_scaled = self.price_scaler.transform(price_feats)
            sentiment_scaled = self.sentiment_scaler.transform(sentiment_feats)
            targets_scaled = self.target_scaler.transform(targets.reshape(-1, 1)).flatten()
        
        # Create sequences
        X_price, X_sentiment, y = [], [], []
        for i in range(len(targets_scaled) - seq_len):
            X_price.append(price_scaled[i:i+seq_len])
            X_sentiment.append(sentiment_scaled[i:i+seq_len])
            y.append(targets_scaled[i+seq_len])
        
        return np.array(X_price), np.array(X_sentiment), np.array(y)

print("DataProcessor class đã được định nghĩa hoàn chỉnh.")

DataProcessor class đã được định nghĩa hoàn chỉnh.


## Metrics & Visualize

In [7]:
def get_performance_metrics(y_true, y_pred):
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    mse   = mean_squared_error(y_true, y_pred)
    rmse  = np.sqrt(mse)
    mae   = mean_absolute_error(y_true, y_pred)
    mape  = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2    = r2_score(y_true, y_pred)
    true_diff = np.diff(y_true)
    pred_diff = y_pred[1:] - y_true[:-1]
    da    = np.mean((true_diff * pred_diff) > 0) * 100
    return {"MSE": mse, "RMSE": rmse, "MAE": mae, "MAPE (%)": mape, "R2": r2, "DA (%)": da}


def train_model(model, train_loader, val_loader, stock_name,
                epochs=150, is_sentiment=False, learning_rate=None):
    if learning_rate is None:
        learning_rate = SENTIMENT_LEARNING_RATE if is_sentiment else LEARNING_RATE

    optimizer  = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler  = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=15, min_lr=1e-6, threshold=1e-4)
    criterion  = nn.MSELoss()
    history    = {"train_loss": [], "val_loss": []}
    best_val   = float('inf')
    patience_counter = 0
    best_epoch = 0
    patience   = 25

    mode_name = "Sentiment Mode" if is_sentiment else "Hybrid Mode"
    print(f"Training {stock_name} ({mode_name})...")

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0
        for p, s, y in train_loader:
            p, s, y = p.to(DEVICE), s.to(DEVICE), y.to(DEVICE).unsqueeze(1)
            optimizer.zero_grad()
            loss = criterion(model(p, s), y)
            loss.backward(); optimizer.step()
            train_loss += loss.item()

        model.eval()
        v_loss = 0
        with torch.no_grad():
            for p, s, y in val_loader:
                p, s, y = p.to(DEVICE), s.to(DEVICE), y.to(DEVICE).unsqueeze(1)
                v_loss += criterion(model(p, s), y).item()

        tl = train_loss / len(train_loader)
        vl = v_loss    / len(val_loader)
        history["train_loss"].append(tl)
        history["val_loss"].append(vl)
        scheduler.step(vl)

        if vl < best_val - 1e-4:
            best_val = vl; patience_counter = 0; best_epoch = epoch
        else:
            patience_counter += 1

        if epoch % 20 == 0 or epoch == 1:
            print(f"Epoch {epoch}/{epochs} - Train: {tl:.6f}, Val: {vl:.6f}")

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch} (best: {best_epoch})")
            break

    save_path = os.path.join(LOG_DIR, f"{stock_name}_Sentiment.pt")
    torch.save(model.state_dict(), save_path)
    print(f"  Model saved → {save_path}")
    return history, save_path


def visualize_prediction(model, X_p_tensor, X_s_tensor, y_tensor,
                          stock_name, dates, mode, filename, processor):
    model.eval()
    with torch.no_grad():
        preds_scaled   = model(X_p_tensor.to(DEVICE), X_s_tensor.to(DEVICE)).squeeze().cpu().numpy()
        targets_scaled = y_tensor.cpu().numpy()
        predictions = processor.target_scaler.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
        targets     = processor.target_scaler.inverse_transform(targets_scaled.reshape(-1, 1)).flatten()

    plot_dates  = pd.to_datetime(dates)
    source_name = filename.replace('.csv', '')
    date_fmt    = mdates.DateFormatter('%d/%m')

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    plt.subplots_adjust(hspace=0.4, wspace=0.25)

    axes[0, 0].plot(plot_dates, targets,     label='Thực tế', color='blue', alpha=0.5, linewidth=1)
    axes[0, 0].plot(plot_dates, predictions, label='Dự báo',  color='red',  linestyle='--', alpha=0.8)
    axes[0, 0].set_title(f'Dự báo Tổng thể ({stock_name}) - {mode}')
    axes[0, 0].xaxis.set_major_formatter(date_fmt)
    axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].scatter(targets, predictions, alpha=0.5, color='purple', s=10)
    axes[0, 1].plot([targets.min(), targets.max()], [targets.min(), targets.max()], 'k--', lw=1)
    axes[0, 1].set_title('Tương quan Thực tế vs Dự báo')
    axes[0, 1].set_xlabel('Giá Thực tế'); axes[0, 1].set_ylabel('Giá Dự báo')

    last_n = 5
    axes[1, 0].plot(plot_dates[-last_n:], targets[-last_n:],     marker='o', label='Thực tế', color='blue')
    axes[1, 0].plot(plot_dates[-last_n:], predictions[-last_n:], marker='x', label='Dự báo',  color='red', linestyle='--')
    axes[1, 0].set_title(f'{last_n} ngày cuối cùng')
    axes[1, 0].xaxis.set_major_formatter(date_fmt)
    axes[1, 0].legend(); axes[1, 0].grid(True)

    error_pct = np.abs((targets - predictions) / (targets + 1e-9)) * 100
    axes[1, 1].bar(plot_dates[-20:], error_pct[-20:], color='orange', alpha=0.6)
    axes[1, 1].set_title('Sai số (%) - 20 ngày cuối')
    axes[1, 1].xaxis.set_major_formatter(date_fmt)
    axes[1, 1].set_ylabel('% Error')

    fig.autofmt_xdate()
    plot_filename = f"{source_name}_{mode}_2x2.png"
    plt.savefig(os.path.join(CHART_DIR, plot_filename), bbox_inches='tight', dpi=150)
    plt.show(); plt.close(fig)

    m = get_performance_metrics(targets, predictions)
    return {
        "Ticker": stock_name, "Mode": mode,
        "MSE": m["MSE"], "RMSE": m["RMSE"], "MAE": m["MAE"],
        "MAPE (%)": m["MAPE (%)"], "R2": m["R2"], "DA (%)": m["DA (%)"],
        "Actual Last": targets[-1], "Pred Last": predictions[-1]
    }

print("Hàm utilities (train_model, visualize_prediction, get_performance_metrics) đã sẵn sàng.")

Hàm utilities (train_model, visualize_prediction, get_performance_metrics) đã sẵn sàng.


---
## CELL 5 — TRAIN MODEL
Đọc dữ liệu từ `DATASET/TRAIN/PRICE` và `DATASET/TRAIN/SENTIMENT`, train model và lưu `.pt`

In [6]:
# =============================================================================
# CELL 5: TRAIN MODEL (3 MODEL TYPES: DLinear, NODE, Hybrid)
#   Nguồn dữ liệu: DATASET/TRAIN/PRICE  +  DATASET/TRAIN/SENTIMENT
#   Chia val 20% từ cuối dữ liệu train (không phải test set)
#   Lưu model .pt + scaler vào LOG_DIR (mỗi model_type riêng)
# =============================================================================

import pickle   # để lưu/load scaler

train_all_metrics  = []
trained_processors = {}   # lưu processor (scaler) để dùng ở Cell 6

# Kiểm tra thư mục TRAIN
for d, label in [(TRAIN_PRICE_DIR, 'TRAIN_PRICE'), (TRAIN_SENTIMENT_DIR, 'TRAIN_SENTIMENT')]:
    if not os.path.exists(d):
        print(f"[WARNING] Thư mục không tồn tại: {d}  ({label})")
    else:
        n = len([f for f in os.listdir(d) if f.endswith('.csv')])
        print(f"[OK] {label}: {n} file CSV tại {d}")

# ── Lấy danh sách file train ──────────────────────────────────────────────
train_price_files     = [f for f in os.listdir(TRAIN_PRICE_DIR)     if f.endswith('.csv')]
train_sentiment_files = [f for f in os.listdir(TRAIN_SENTIMENT_DIR) if f.endswith('.csv')]

# Lọc ticker duy nhất
seen_tickers, filtered_train_files = set(), []
for f in train_price_files:
    t = f.split('_')[0].upper()
    if t not in seen_tickers:
        filtered_train_files.append(f)
        seen_tickers.add(t)

print(f"\nBắt đầu TRAIN {len(filtered_train_files)} mã cổ phiếu × {len(MODEL_TYPES)} model types...")
print(f"Model types: {MODEL_TYPES}")
print("="*80)

for model_type in MODEL_TYPES:
    print(f"\n{'='*80}")
    print(f" TRAINING: Model Type = {model_type.upper()}")
    print(f"{'='*80}")

    for filename in filtered_train_files:
        ticker_core    = os.path.splitext(filename)[0].split('_')[0].upper()
        sentiment_file = next((s for s in train_sentiment_files if ticker_core in s.upper()), None)

        if sentiment_file is None:
            print(f"[SKIP] {ticker_core}: Không tìm thấy sentiment file trong TRAIN.")
            continue

        print(f"\n>>> TRAIN [{model_type.upper()}]: {ticker_core}")
        try:
            processor = DataProcessor()
            df_raw = processor.load_and_merge_data(
                os.path.join(TRAIN_PRICE_DIR, filename),
                os.path.join(TRAIN_SENTIMENT_DIR, sentiment_file)
            )

            # Không dropna - để create_features chọn cột sentiment thực
            df_merged = df_raw.copy()

            # ✅ MỚI: Sử dụng tất cả dữ liệu, dù ít đi nữa (động thích ứng SEQ_LEN)
            actual_seq_len = min(SEQ_LEN, max(1, len(df_merged) - 1))  # Ít nhất 1 sequence
            if actual_seq_len < SEQ_LEN:
                print(f"  [INFO] Dữ liệu TRAIN nhỏ ({len(df_merged)} dòng): điều chỉnh seq_len từ {SEQ_LEN} → {actual_seq_len}")
            else:
                actual_seq_len = SEQ_LEN

            # Feature Engineering + tạo sequence (fit=True → fit scaler trên train)
            price_f, sent_f, targets = processor.create_features(df_merged)
            
            if model_type in ['dlinear', 'node']:
                # DLinear & NODE: Chỉ dùng price, không cần sentiment
                print(f"  [INFO] {model_type.upper()}: Chỉ dùng Price ({len(price_f.columns)} cột)")
                X_p, X_s, y = processor.prepare_sequences(
                    price_f, price_f.copy() * 0, targets, ticker_core, actual_seq_len, fit=True)  # X_s dummy
            else:
                # Hybrid: Dùng cả price và sentiment
                print(f"  [INFO] Sentiment columns ({len(sent_f.columns)}): {list(sent_f.columns)}")
                X_p, X_s, y = processor.prepare_sequences(
                    price_f, sent_f, targets, ticker_core, actual_seq_len, fit=True)

            # Chia train/val từ dữ liệu TRAIN (80/20)
            split = int(len(y) * 0.8)
            X_p_tr, X_p_vl = torch.FloatTensor(X_p[:split]), torch.FloatTensor(X_p[split:])
            X_s_tr, X_s_vl = torch.FloatTensor(X_s[:split]), torch.FloatTensor(X_s[split:])
            y_tr,   y_vl   = torch.FloatTensor(y[:split]),   torch.FloatTensor(y[split:])

            # Khởi tạo model theo type
            price_dim = X_p.shape[2]
            sentiment_dim = X_s.shape[2]
            model = get_model_by_type(
                model_type, 
                seq_len=actual_seq_len,
                price_dim=price_dim,
                sentiment_dim=sentiment_dim, 
                hidden_dim=SENTIMENT_HIDDEN_DIM, 
                dropout=SENTIMENT_DROPOUT
            ).to(DEVICE)

            model_save_name = f"{ticker_core}_{model_type.upper()}"
            history, saved_path = train_model(
                model,
                DataLoader(TensorDataset(X_p_tr, X_s_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True),
                DataLoader(TensorDataset(X_p_vl, X_s_vl, y_vl), batch_size=BATCH_SIZE),
                model_save_name,
                epochs=SENTIMENT_EPOCHS,
                is_sentiment=True,
                learning_rate=SENTIMENT_LEARNING_RATE
            )

            # Lưu scaler (để dùng lại khi inference TEST)
            scaler_path = os.path.join(LOG_DIR, f"{ticker_core}_{model_type}_scaler.pkl")
            with open(scaler_path, 'wb') as f_pkl:
                pickle.dump({
                    'price_scaler':    processor.price_scaler,
                    'sentiment_scaler': processor.sentiment_scaler,
                    'target_scaler':   processor.target_scaler,
                    'price_cols':      list(price_f.columns),
                    'sent_cols':       list(sent_f.columns),
                    'price_dim':       price_dim,
                    'sentiment_dim':   sentiment_dim
                }, f_pkl)
            print(f"  Scaler saved → {scaler_path}")

            # Lưu processor vào dict để dùng liền ở Cell 6 (nếu chạy cùng session)
            trained_processors[f"{ticker_core}_{model_type}"] = processor

            # Visualize trên val set của TRAIN
            model.load_state_dict(torch.load(os.path.abspath(saved_path), map_location=DEVICE))
            val_dates = df_merged['Date'].values[split + actual_seq_len:]
            metrics = visualize_prediction(
                model=model,
                X_p_tensor=X_p_vl, X_s_tensor=X_s_vl, y_tensor=y_vl,
                stock_name=ticker_core, dates=val_dates,
                mode=f'TRAIN_val_{model_type.upper()}', filename=filename, processor=processor
            )
            if metrics:
                metrics['Model_Type'] = model_type
                train_all_metrics.append(metrics)

        except Exception as e:
            print(f"  Lỗi tại {ticker_core}: {e}")
            import traceback; traceback.print_exc()

# Bảng tổng hợp sau TRAIN
if train_all_metrics:
    print("\n" + "="*80)
    print(" KẾT QUẢ VALIDATION SAU TRAIN (CẢ 3 MODEL TYPE)")
    print("="*80)
    train_summary = pd.DataFrame(train_all_metrics)
    display_cols = ['Ticker', 'Model_Type', 'Mode', 'DA (%)', 'MAPE (%)', 'RMSE', 'MAE', 'R2']
    display(train_summary[display_cols].round(4))
    train_summary.to_csv(os.path.join(LOG_DIR, 'train_summary.csv'), index=False, encoding='utf-8-sig')
    print(f"Đã lưu train_summary.csv")

else:    print("Không có kết quả train để hiển thị!")

[OK] TRAIN_PRICE: 6 file CSV tại ..\..\DATASET\TRAIN\PRICE
[OK] TRAIN_SENTIMENT: 12 file CSV tại ..\..\DATASET\TRAIN\SENTIMENT

Bắt đầu TRAIN 6 mã cổ phiếu × 3 model types...
Model types: ['dlinear', 'node', 'hybrid']

 TRAINING: Model Type = DLINEAR

>>> TRAIN [DLINEAR]: ALIBABA
 THỐNG KÊ FILE GỐC:
   Price     : 2558 dòng
   Sentiment : 164 dòng
    [WARNING] No sentiment columns found!
  [INFO] DLINEAR: Chỉ dùng Price (4 cột)
  Lỗi tại ALIBABA: name 'train_model' is not defined

>>> TRAIN [DLINEAR]: AMAZON


Traceback (most recent call last):
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_624\3665590054.py", line 101, in <module>
    history, saved_path = train_model(
                          ^^^^^^^^^^^
NameError: name 'train_model' is not defined
Traceback (most recent call last):
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_624\3665590054.py", line 101, in <module>
    history, saved_path = train_model(
                          ^^^^^^^^^^^
NameError: name 'train_model' is not defined


 THỐNG KÊ FILE GỐC:
   Price     : 2860 dòng
   Sentiment : 166 dòng
    [WARNING] No sentiment columns found!
  [INFO] DLINEAR: Chỉ dùng Price (4 cột)
  Lỗi tại AMAZON: name 'train_model' is not defined

>>> TRAIN [DLINEAR]: APPLE
 THỐNG KÊ FILE GỐC:
   Price     : 2860 dòng
   Sentiment : 251 dòng
    [WARNING] No sentiment columns found!
  [INFO] DLINEAR: Chỉ dùng Price (4 cột)
  Lỗi tại APPLE: name 'train_model' is not defined

>>> TRAIN [DLINEAR]: GOOGLE


Traceback (most recent call last):
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_624\3665590054.py", line 101, in <module>
    history, saved_path = train_model(
                          ^^^^^^^^^^^
NameError: name 'train_model' is not defined


 THỐNG KÊ FILE GỐC:
   Price     : 2860 dòng
   Sentiment : 787 dòng
    [WARNING] No sentiment columns found!
  [INFO] DLINEAR: Chỉ dùng Price (4 cột)
  Lỗi tại GOOGLE: name 'train_model' is not defined

>>> TRAIN [DLINEAR]: META


...

Traceback (most recent call last):
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_624\3665590054.py", line 101, in <module>
    history, saved_path = train_model(
                          ^^^^^^^^^^^
NameError: name 'train_model' is not defined


 THỐNG KÊ FILE GỐC:
   Price     : 3204 dòng
   Sentiment : 831 dòng
    [WARNING] No sentiment columns found!
  [INFO] Sentiment columns (1): ['placeholder_sentiment']
  Lỗi tại META: name 'train_model' is not defined

>>> TRAIN [HYBRID]: VNM


Traceback (most recent call last):
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_624\3665590054.py", line 101, in <module>
    history, saved_path = train_model(
                          ^^^^^^^^^^^
NameError: name 'train_model' is not defined


 THỐNG KÊ FILE GỐC:
   Price     : 2859 dòng
   Sentiment : 1166 dòng
    [WARNING] No sentiment columns found!
  [INFO] Sentiment columns (1): ['placeholder_sentiment']
  Lỗi tại VNM: name 'train_model' is not defined
Không có kết quả train để hiển thị!


Traceback (most recent call last):
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_624\3665590054.py", line 101, in <module>
    history, saved_path = train_model(
                          ^^^^^^^^^^^
NameError: name 'train_model' is not defined


---
## CELL 6 — TEST / INFERENCE + DỰ BÁO 5 NGÀY TƯƠNG LAI
- Đọc dữ liệu từ `DATASET/TEST/PRICE` và `DATASET/TEST/SENTIMENT`
- Gọi model đã train (`.pt`) và scaler đã lưu (`.pkl`)
- Đánh giá trên test set
- Xuất dự báo 5 ngày tương lai vào `{TICKER}_predict.csv`

In [27]:
def export_predict_csv(ticker_core, df_merged, model, processor, device, future_days=FUTURE_DAYS, model_type='hybrid'):
    """
    Xuất file {ticker_core}_{model_type}_predict.csv gồm:
      - Dự báo trên toàn bộ TEST (Type='Predicted')
      - Dự báo N ngày tương lai (Type='Future')
    """
    model.eval()

    c_name = get_col_name(df_merged, ['Lần cuối', 'Close', 'Adj Close'])
    o_name = get_col_name(df_merged, ['Mở', 'Open'])
    h_name = get_col_name(df_merged, ['Cao', 'High'])
    l_name = get_col_name(df_merged, ['Thấp', 'Low'])
    v_name = get_col_name(df_merged, ['Khối lượng', 'Volume', 'Vol'])

    if c_name is None:
        print(f"[SKIP] {ticker_core}: Không tìm thấy cột giá. Cột: {list(df_merged.columns)}")
        return None

    # Tỷ lệ OHLCV từ 30 ngày gần nhất của TEST
    recent  = df_merged.tail(30).copy()
    h_ratio = (recent[h_name] / recent[c_name]).mean()          if h_name else 1.01
    l_ratio = (recent[l_name] / recent[c_name]).mean()          if l_name else 0.99
    o_ratio = (recent[o_name] / recent[c_name].shift(1)).mean() if o_name else 1.00
    avg_vol = recent[v_name].mean()                              if v_name else 0
    h_ratio = 1.01 if np.isnan(h_ratio) else h_ratio
    l_ratio = 0.99 if np.isnan(l_ratio) else l_ratio
    o_ratio = 1.00 if np.isnan(o_ratio) else o_ratio

    # Lấy features rồi transform (KHÔNG fit — scaler đã fit từ TRAIN)
    # QUAN TRỌNG: Pass processor.sentiment_cols để ensure dims match
    price_f, sent_f, _ = processor.create_features(df_merged, specific_sent_cols=processor.sentiment_cols)
    p_all = processor.price_scaler.transform(price_f.values.astype('float32'))
    s_all = processor.sentiment_scaler.transform(sent_f.values.astype('float32'))

    dates_all = pd.to_datetime(df_merged['Date'].values)
    close_all = df_merged[c_name].values.astype(float)

    # ── PHẦN A: Predicted trên toàn bộ TEST ──────────────────────────────
    history_rows = []
    with torch.no_grad():
        for t in range(SEQ_LEN, len(df_merged)):
            x_p = torch.FloatTensor(p_all[t - SEQ_LEN:t]).unsqueeze(0).to(device)
            x_s = torch.FloatTensor(s_all[t - SEQ_LEN:t]).unsqueeze(0).to(device)
            pred_scaled = model(x_p, x_s).cpu().item()
            close_pred  = processor.target_scaler.inverse_transform([[pred_scaled]])[0][0]
            prev_close  = close_all[t - 1]
            open_pred   = prev_close * o_ratio
            high_pred   = max(close_pred * h_ratio, open_pred, close_pred)
            low_pred    = min(close_pred * l_ratio, open_pred, close_pred)
            history_rows.append({
                'Date'   : dates_all[t].strftime('%Y-%m-%d'),
                'Open'   : round(float(open_pred),  2),
                'High'   : round(float(high_pred),  2),
                'Low'    : round(float(low_pred),   2),
                'Close'  : round(float(close_pred), 2),
                'Actual' : round(float(close_all[t]), 2),
                'Volume' : int(avg_vol),
                'Type'   : 'Predicted'
            })

    print(f"   → Predicted trên TEST : {len(history_rows)} dòng")

    # ── PHẦN B: Dự báo N ngày TƯƠNG LAI (sliding window) ─────────────────
    current_p       = list(p_all[-SEQ_LEN:])
    current_s       = list(s_all[-SEQ_LEN:])
    last_close_real = float(close_all[-1])
    last_date       = dates_all[-1]

    future_rows = []
    with torch.no_grad():
        for day_i in range(future_days):
            x_p = torch.FloatTensor(np.array(current_p[-SEQ_LEN:])).unsqueeze(0).to(device)
            x_s = torch.FloatTensor(np.array(current_s[-SEQ_LEN:])).unsqueeze(0).to(device)
            pred_scaled = model(x_p, x_s).cpu().item()
            close_pred  = processor.target_scaler.inverse_transform([[pred_scaled]])[0][0]
            open_pred   = last_close_real * o_ratio
            high_pred   = max(close_pred * h_ratio, open_pred, close_pred)
            low_pred    = min(close_pred * l_ratio, open_pred, close_pred)

            # Bỏ qua T7, CN → ngày làm việc tiếp theo
            target_date = last_date + timedelta(days=1)
            while target_date.weekday() >= 5:
                target_date += timedelta(days=1)

            future_rows.append({
                'Date'   : target_date.strftime('%Y-%m-%d'),
                'Open'   : round(float(open_pred),  2),
                'High'   : round(float(high_pred),  2),
                'Low'    : round(float(low_pred),   2),
                'Close'  : round(float(close_pred), 2),
                'Actual' : None,   # chưa có thực tế
                'Volume' : int(avg_vol),
                'Type'   : 'Future'
            })

            # Cập nhật sliding window
            row_map = {
                'Open': open_pred, 'High': high_pred, 'Low': low_pred, 'Close': close_pred,
                'Mở': open_pred,   'Cao':  high_pred, 'Thấp': low_pred, 'Lần cuối': close_pred
            }
            new_row_vec = [row_map.get(col, close_pred) for col in price_f.columns]
            current_p.append(processor.price_scaler.transform([new_row_vec])[0])
            current_s.append(s_all[-1])
            last_close_real = close_pred
            last_date       = target_date

    print(f"   → Future {future_days} ngày   : {len(future_rows)} dòng")

    # ── GHÉP + XUẤT FILE ─────────────────────────────────────────────────
    predict_df = pd.DataFrame(history_rows + future_rows)[
        ['Date', 'Open', 'High', 'Low', 'Close', 'Actual', 'Volume', 'Type']
    ]

    out_file = os.path.join(LOG_DIR, f'{ticker_core}_{model_type}_predict.csv')
    predict_df.to_csv(out_file, index=False, encoding='utf-8-sig')
    print(f"   ✓ File lưu tại: {out_file}")

    # ── IN BẢNG 5 NGÀY TƯƠNG LAI ─────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f" DỰ BÁO {future_days} NGÀY TƯƠNG LAI — {ticker_core} ({model_type.upper()})")
    print(f"{'─'*50}")
    future_df = predict_df[predict_df['Type'] == 'Future'][['Date', 'Open', 'High', 'Low', 'Close']]
    print(future_df.to_string(index=False))

    return predict_df

In [28]:
# =============================================================================
# HELPER: Ensure processor.sentiment_cols is properly set when loading from pickle
# This code will be inserted to handle sentiment column dimension mismatches
# =============================================================================

def ensure_processor_sentiment_cols(processor, scaler_pickle_path, ticker_core):
    """
    Ensure processor.sentiment_cols is set from the scaler pickle file.
    This prevents dimension mismatches when export_predict_csv calls create_features.
    """
    if processor.sentiment_cols is None and os.path.exists(scaler_pickle_path):
        try:
            with open(scaler_pickle_path, 'rb') as f:
                saved = pickle.load(f)
            processor.sentiment_cols = saved.get('sent_cols', None)
            if processor.sentiment_cols:
                print(f"  [FIX] Set processor.sentiment_cols = {processor.sentiment_cols}")
        except Exception as e:
            print(f"  [WARNING] Failed to load sentiment_cols from pickle: {e}")
    return processor


In [3]:
class DataProcessor:
    def __init__(self):
        self.price_scaler    = StandardScaler()
        self.sentiment_scaler = StandardScaler()
        self.target_scaler   = StandardScaler()
        self.sentiment_cols  = None  # For TEST phase: track which cols to use

    def load_and_merge_data(self, price_path, sentiment_path):
        """Load and merge price + sentiment data with robust date handling."""
        price_df = pd.read_csv(price_path,     encoding='utf-8-sig')
        sent_df  = pd.read_csv(sentiment_path, encoding='utf-8-sig')

        for df in [price_df, sent_df]:
            # Find and rename date column
            date_col = next((c for c in df.columns if c.lower() in ['ngày', 'date', 'time']), None)
            if date_col:
                df.rename(columns={date_col: 'Date'}, inplace=True)
            
            # Parse dates robustly (handles both YYYY-MM-DD and MM/DD/YYYY formats)
            df['Date'] = pd.to_datetime(df['Date'], errors='coerce', format='mixed')
            # Normalize to date only (ignore time part)
            df['Date'] = df['Date'].dt.normalize()

        price_df = price_df.dropna(subset=['Date']).drop_duplicates('Date').sort_values('Date')
        sent_df  = sent_df.dropna(subset=['Date'])

        sent_cols_to_process = [c for c in sent_df.columns if any(
            w in c.lower() for w in ['score', 'impact', 'relevance', 'sentiment',
                                     'confidence', 'momentum', 'volatility', 'prob'])]
        for col in sent_cols_to_process:
            sent_df[col] = pd.to_numeric(sent_df[col], errors='coerce')
        sent_df = sent_df.groupby('Date')[sent_cols_to_process].mean().reset_index()

        print(f" THỐNG KÊ FILE GỐC:")
        print(f"   Price     : {len(price_df)} dòng")
        print(f"   Sentiment : {len(sent_df)} dòng")

        merged_df = pd.merge(price_df, sent_df, on='Date', how='left')
        return merged_df

    def create_features(self, df, specific_sent_cols=None):
        """Extract price and sentiment features."""
        # ── PRICE FEATURES ───────────────────────────────────────────────
        price_cols = [c for c in ['Lần cuối', 'Mở', 'Cao', 'Thấp',
                                   'Close', 'Open', 'High', 'Low'] if c in df.columns]
        price_feats = df[price_cols].copy()
        for col in price_cols:
            if price_feats[col].dtype == 'object':
                price_feats[col] = price_feats[col].str.replace(',', '').astype(float)
        price_feats = price_feats.ffill().bfill().fillna(0)

        # ── SENTIMENT FEATURES ───────────────────────────────────────────
        if specific_sent_cols is not None:
            sent_cols_to_use = [c for c in specific_sent_cols if c in df.columns]
            if not sent_cols_to_use:
                print(f"    [WARNING] None of specified sentiment cols found: {specific_sent_cols}")
                sent_cols_to_use = []
        else:
            standard_sent_cols = [
                'sentiment_score', 'impact_score', 'relevance_score', 'confidence',
                'short_term_score', 'medium_term_score', 'sentiment_momentum',
                'sentiment_volatility', 'sentiment_trend'
            ]
            sent_cols_to_use = [c for c in standard_sent_cols if c in df.columns]
            
            if len(sent_cols_to_use) < 9:
                sent_cols_to_use = [c for c in df.columns if any(
                    word in c.lower() for word in 
                    ['score', 'impact', 'relevance', 'sentiment', 'confidence',
                     'momentum', 'volatility', 'prob'])]

        if not sent_cols_to_use:
            print(f"    [WARNING] No sentiment columns found!")
            sent_feats = pd.DataFrame(0.0, index=df.index, columns=['placeholder_sentiment'])
        else:
            sent_feats = df[sent_cols_to_use].copy().fillna(0)
            if self.sentiment_cols is None:
                self.sentiment_cols = sent_cols_to_use

        target_col = next((c for c in ['Lần cuối', 'Close'] if c in df.columns), price_cols[0] if price_cols else 'Close')
        return price_feats, sent_feats, df[target_col].values

    def prepare_sequences(self, price_feats, sent_feats, targets, stock_name, seq_len, fit=True):
        """Transform features into sequences. fit=True for TRAIN, fit=False for TEST."""
        p_data = price_feats.values.astype('float32')
        s_data = sent_feats.values.astype('float32')
        t_data = targets.astype('float32').reshape(-1, 1)

        if fit:
            p_scaled = self.price_scaler.fit_transform(p_data)
            s_scaled = self.sentiment_scaler.fit_transform(s_data)
            t_scaled = self.target_scaler.fit_transform(t_data).flatten()
        else:
            p_scaled = self.price_scaler.transform(p_data)
            s_scaled = self.sentiment_scaler.transform(s_data)
            t_scaled = self.target_scaler.transform(t_data).flatten()

        X_p, X_s, y = [], [], []
        for i in range(len(targets) - seq_len):
            X_p.append(p_scaled[i:i + seq_len])
            X_s.append(s_scaled[i:i + seq_len])
            y.append(t_scaled[i + seq_len])

        if not X_p:
            raise ValueError("Không tạo được sequence nào.")

        return np.array(X_p), np.array(X_s), np.array(y)

print("✓ DataProcessor (FIXED) được định nghĩa - hỗ trợ cả YYYY-MM-DD và MM/DD/YYYY")

✓ DataProcessor (FIXED) được định nghĩa - hỗ trợ cả YYYY-MM-DD và MM/DD/YYYY


In [4]:
# =============================================================================
# CELL 6: TEST / INFERENCE (3 MODEL TYPES: DLinear, NODE, Hybrid)
#   Nguồn dữ liệu: DATASET/TEST/PRICE  +  DATASET/TEST/SENTIMENT
#   Load model .pt + scaler .pkl từ TRAIN (mỗi model_type riêng)
#   Đánh giá + lưu dự báo & metrics
# =============================================================================

import pickle
from datetime import timedelta

def get_col_name(df, candidates):
    """Tìm cột tương ứng (ưu tiên danh sách đầu tiên)."""
    for col in candidates:
        if col in df.columns:
            return col
    return None

# Kiểm tra thư mục TEST
for d, label in [(TEST_PRICE_DIR, 'TEST_PRICE'), (TEST_SENTIMENT_DIR, 'TEST_SENTIMENT')]:
    if not os.path.exists(d):
        print(f"[WARNING] Thư mục không tồn tại: {d}  ({label})")
    else:
        n = len([f for f in os.listdir(d) if f.endswith('.csv')])
        print(f"[OK] {label}: {n} file CSV tại {d}")

# ── Lấy danh sách file test ──────────────────────────────────────────────
test_price_files     = [f for f in os.listdir(TEST_PRICE_DIR)     if f.endswith('.csv')]
test_sentiment_files = [f for f in os.listdir(TEST_SENTIMENT_DIR) if f.endswith('.csv')]

# Lọc ticker duy nhất
seen_test_tickers, filtered_test_files = set(), []
for f in test_price_files:
    t = f.split('_')[0].upper()
    if t not in seen_test_tickers:
        filtered_test_files.append(f)
        seen_test_tickers.add(t)

print(f"\nBắt đầu TEST/INFERENCE {len(filtered_test_files)} mã cổ phiếu × {len(MODEL_TYPES)} model types...")
print(f"Model types: {MODEL_TYPES}")
print("="*80)

test_all_metrics = []

for model_type in MODEL_TYPES:
    print(f"\n{'='*80}")
    print(f" TESTING: Model Type = {model_type.upper()}")
    print(f"{'='*80}")

    for filename in filtered_test_files:
        ticker_core    = os.path.splitext(filename)[0].split('_')[0].upper()
        sentiment_file = next((s for s in test_sentiment_files if ticker_core in s.upper()), None)

        if sentiment_file is None:
            print(f"\n[SKIP] {ticker_core}: Không tìm thấy sentiment file trong TEST.")
            continue

        print(f"\n>>> TEST [{model_type.upper()}]: {ticker_core}")
        try:
            # ── BƯỚC 1: Load scaler từ pickle được lưu lúc TRAIN ───────────────
            scaler_pickle_path = os.path.join(LOG_DIR, f"{ticker_core}_{model_type}_scaler.pkl")
            if not os.path.exists(scaler_pickle_path):
                print(f"  [SKIP] Scaler file not found: {scaler_pickle_path}")
                continue

            with open(scaler_pickle_path, 'rb') as f_pkl:
                saved_scaler_data = pickle.load(f_pkl)
            
            price_scaler    = saved_scaler_data.get('price_scaler')
            sentiment_scaler= saved_scaler_data.get('sentiment_scaler')
            target_scaler   = saved_scaler_data.get('target_scaler')
            sent_cols_from_train = saved_scaler_data.get('sent_cols', [])
            price_dim       = saved_scaler_data.get('price_dim', 5)
            sentiment_dim   = saved_scaler_data.get('sentiment_dim', 1)

            print(f"  [OK] Scaler loaded: price_dim={price_dim}, sentiment_dim={sentiment_dim}")

            # ── BƯỚC 2: Khởi tạo processor và gắn scaler ──────────────────────
            processor = DataProcessor()
            processor.price_scaler    = price_scaler
            processor.sentiment_scaler= sentiment_scaler
            processor.target_scaler   = target_scaler
            processor.sentiment_cols  = sent_cols_from_train if sent_cols_from_train != ['placeholder_sentiment'] else None

            # ── BƯỚC 3: Load và merge test data ───────────────────────────────
            df_test = processor.load_and_merge_data(
                os.path.join(TEST_PRICE_DIR, filename),
                os.path.join(TEST_SENTIMENT_DIR, sentiment_file)
            )

            # ✅ MỚI: Sử dụng tất cả dữ liệu, dù ít đi nữa (động thích ứng SEQ_LEN)
            actual_seq_len = min(SEQ_LEN, max(1, len(df_test) - 1))  # Ít nhất 1 sequence
            if actual_seq_len < SEQ_LEN:
                print(f"  [INFO] Dữ liệu TEST nhỏ ({len(df_test)} dòng): điều chỉnh seq_len từ {SEQ_LEN} → {actual_seq_len}")
            else:
                actual_seq_len = SEQ_LEN

            # ── BƯỚC 4: Tạo features (dùng transform, KHÔNG fit lại scaler) ────
            if model_type in ['dlinear', 'node']:
                # DLinear & NODE: Chỉ dùng price
                price_f_test, sent_f_test, targets_test = processor.create_features(df_test, specific_sent_cols=None)
                # Tạo X_s dummy VỚI ĐÚNG SỐ CỘT như scaler được fit
                # Điều này tránh mismatch dimension khi transform
                sent_f_test_real = pd.DataFrame(
                    0.0, 
                    index=sent_f_test.index, 
                    columns=[f'dummy_sent_{i}' for i in range(sentiment_dim)]
                )
                print(f"    [INFO] {model_type.upper()}: Chỉ dùng Price (dummy sentiment: {sentiment_dim} cols)")
            else:
                # Hybrid: Dùng cả price và sentiment
                if sent_cols_from_train == ['placeholder_sentiment']:
                    cols_to_pass = None
                    print(f"    [INFO] Placeholder detected in TRAIN scaler → allowing auto-discovery in TEST")
                else:
                    cols_to_pass = sent_cols_from_train
                    print(f"    [INFO] Using TRAIN sentiment columns: {cols_to_pass}")
                
                price_f_test, sent_f_test_real, targets_test = processor.create_features(
                    df_test, specific_sent_cols=cols_to_pass)
                sent_f_test = sent_f_test_real

            # Kiểm tra shape khớp với scaler
            if price_f_test.shape[1] != price_dim:
                print(f"    [WARNING] Price dim mismatch: test={price_f_test.shape[1]} vs train={price_dim}")
            
            if sent_f_test.shape[1] != sentiment_dim and model_type == 'hybrid':
                if sentiment_dim == 1 and sent_f_test.shape[1] > 1:
                    print(f"    [INFO] Placeholder→Real sentiment cols mismatch detected")
                    processor.sentiment_scaler = StandardScaler()
                    processor.sentiment_scaler.fit(sent_f_test.values.astype('float32'))
                    print(f"           Refitted sentiment_scaler for {sent_f_test.shape[1]} features")
                else:
                    print(f"    [WARNING] Sentiment dim mismatch: test={sent_f_test.shape[1]} vs train={sentiment_dim}")

            X_p_test, X_s_test, y_test = processor.prepare_sequences(
                price_f_test, sent_f_test, targets_test,
                ticker_core, actual_seq_len, fit=False  # ← QUAN TRỌNG: fit=False
            )

            # ── BƯỚC 5: Load model đã train ───────────────────────────────────
            model = get_model_by_type(
                model_type,
                seq_len=actual_seq_len,
                price_dim=price_f_test.shape[1],  # DataFrame columns
                sentiment_dim=sent_f_test.shape[1],  # DataFrame columns
                hidden_dim=SENTIMENT_HIDDEN_DIM,
                dropout=SENTIMENT_DROPOUT
            ).to(DEVICE)

            model_path = os.path.join(LOG_DIR, f"{ticker_core}_{model_type}.pt")
            if not os.path.exists(model_path):
                print(f"  [SKIP] Model file not found: {model_path}")
                continue

            model.load_state_dict(torch.load(model_path, map_location=DEVICE))
            print(f"  [OK] Model loaded: {model_path}")

            # ── BƯỚC 6: Đánh giá trên test set ────────────────────────────────
            model.eval()
            X_p_test_t = torch.FloatTensor(X_p_test).to(DEVICE)
            X_s_test_t = torch.FloatTensor(X_s_test).to(DEVICE)
            y_test_t   = torch.FloatTensor(y_test).to(DEVICE)

            with torch.no_grad():
                preds_scaled = model(X_p_test_t, X_s_test_t).squeeze().cpu().numpy()
                targets_scaled = y_test_t.cpu().numpy()

            predictions = processor.target_scaler.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
            targets     = processor.target_scaler.inverse_transform(targets_scaled.reshape(-1, 1)).flatten()

            m = get_performance_metrics(targets, predictions)
            test_metrics = {
                "Ticker": ticker_core, "Model_Type": model_type, "Mode": "TEST",
                "MSE": m["MSE"], "RMSE": m["RMSE"], "MAE": m["MAE"],
                "MAPE (%)": m["MAPE (%)"], "R2": m["R2"], "DA (%)": m["DA (%)"],
                "Actual Last": targets[-1], "Pred Last": predictions[-1]
            }
            test_all_metrics.append(test_metrics)
            print(f"  [METRICS] RMSE={m['RMSE']:.4f}, MAPE={m['MAPE (%)']:.2f}%, R2={m['R2']:.4f}")

            # ── BƯỚC 7: Visualize trên test set ───────────────────────────────
            test_dates = df_test['Date'].values[actual_seq_len:]
            vis_metrics = visualize_prediction(
                model=model,
                X_p_tensor=X_p_test_t, X_s_tensor=X_s_test_t, y_tensor=y_test_t,
                stock_name=ticker_core, dates=test_dates,
                mode=f'TEST_{model_type.upper()}', filename=filename, processor=processor
            )

            # ── BƯỚC 8: Export dự báo CSV ────────────────────────────────────
            export_predict_csv(ticker_core, df_test, model, processor, DEVICE, FUTURE_DAYS, model_type)

        except Exception as e:
            print(f"  Lỗi tại {ticker_core}: {e}")
            import traceback; traceback.print_exc()

# ── Bảng tổng hợp sau TEST ──────────────────────────────────────────────
if test_all_metrics:
    print("\n" + "="*80)
    print(" KẾT QUẢ ĐÁNH GIÁ TRÊN TEST SET (CẢ 3 MODEL TYPE)")
    print("="*80)
    test_summary = pd.DataFrame(test_all_metrics)
    display_cols = ['Ticker', 'Model_Type', 'Mode', 'DA (%)', 'MAPE (%)', 'RMSE', 'MAE', 'R2']
    display(test_summary[display_cols].round(4))
    test_summary.to_csv(os.path.join(LOG_DIR, 'test_evaluation_summary.csv'), index=False, encoding='utf-8-sig')
    print(f"Đã lưu test_evaluation_summary.csv")
else:

    print("Không có kết quả test để hiển thị!")

[OK] TEST_PRICE: 6 file CSV tại ..\..\DATASET\TEST\PRICE
[OK] TEST_SENTIMENT: 8 file CSV tại ..\..\DATASET\TEST\SENTIMENT

Bắt đầu TEST/INFERENCE 6 mã cổ phiếu × 3 model types...
Model types: ['dlinear', 'node', 'hybrid']

 TESTING: Model Type = DLINEAR

>>> TEST [DLINEAR]: ALIBABA
  [OK] Scaler loaded: price_dim=4, sentiment_dim=4
 THỐNG KÊ FILE GỐC:
   Price     : 20 dòng
   Sentiment : 164 dòng
  [INFO] Dữ liệu TEST nhỏ (20 dòng): điều chỉnh seq_len từ 30 → 19
    [WARNING] No sentiment columns found!
    [INFO] DLINEAR: Chỉ dùng Price (dummy sentiment: 4 cols)
  Lỗi tại ALIBABA: X has 1 features, but StandardScaler is expecting 4 features as input.

>>> TEST [DLINEAR]: AMAZON
  [OK] Scaler loaded: price_dim=4, sentiment_dim=4


Traceback (most recent call last):
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_624\3001265464.py", line 135, in <module>
    X_p_test, X_s_test, y_test = processor.prepare_sequences(
                                 ~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        price_f_test, sent_f_test, targets_test,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        ticker_core, actual_seq_len, fit=False  # ← QUAN TRỌNG: fit=False
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_624\971465480.py", line 95, in prepare_sequences
    s_scaled = self.sentiment_scaler.transform(s_data)
  File "d:\NghienCuu\NCT3\.venv\Lib\site-packages\sklearn\utils\_set_output.py", line 316, in wrapped
    data_to_wrap = f(self, X, *args, **kwargs)
  File "d:\NghienCuu\NCT3\.venv\Lib\site-packages\sklearn\preprocessing\_data.py", line 1094, in transform
    X = validate_data(
        self,
    ...<6 lines>...
        ensure_all_f

 THỐNG KÊ FILE GỐC:
   Price     : 20 dòng
   Sentiment : 718 dòng
  [INFO] Dữ liệu TEST nhỏ (20 dòng): điều chỉnh seq_len từ 30 → 19
    [WARNING] No sentiment columns found!
    [INFO] DLINEAR: Chỉ dùng Price (dummy sentiment: 4 cols)
  Lỗi tại AMAZON: X has 1 features, but StandardScaler is expecting 4 features as input.

[SKIP] APPLE: Không tìm thấy sentiment file trong TEST.

>>> TEST [DLINEAR]: GOOGLE
  [OK] Scaler loaded: price_dim=4, sentiment_dim=4
 THỐNG KÊ FILE GỐC:
   Price     : 20 dòng
   Sentiment : 262 dòng
  [INFO] Dữ liệu TEST nhỏ (20 dòng): điều chỉnh seq_len từ 30 → 19
    [INFO] DLINEAR: Chỉ dùng Price (dummy sentiment: 4 cols)
  Lỗi tại GOOGLE: X has 9 features, but StandardScaler is expecting 4 features as input.

>>> TEST [DLINEAR]: META
  [OK] Scaler loaded: price_dim=4, sentiment_dim=4


Traceback (most recent call last):
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_624\3001265464.py", line 135, in <module>
    X_p_test, X_s_test, y_test = processor.prepare_sequences(
                                 ~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        price_f_test, sent_f_test, targets_test,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        ticker_core, actual_seq_len, fit=False  # ← QUAN TRỌNG: fit=False
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_624\971465480.py", line 95, in prepare_sequences
    s_scaled = self.sentiment_scaler.transform(s_data)
  File "d:\NghienCuu\NCT3\.venv\Lib\site-packages\sklearn\utils\_set_output.py", line 316, in wrapped
    data_to_wrap = f(self, X, *args, **kwargs)
  File "d:\NghienCuu\NCT3\.venv\Lib\site-packages\sklearn\preprocessing\_data.py", line 1094, in transform
    X = validate_data(
        self,
    ...<6 lines>...
        ensure_all_f

 THỐNG KÊ FILE GỐC:
   Price     : 20 dòng
   Sentiment : 143 dòng
  [INFO] Dữ liệu TEST nhỏ (20 dòng): điều chỉnh seq_len từ 30 → 19
    [INFO] DLINEAR: Chỉ dùng Price (dummy sentiment: 4 cols)
  Lỗi tại META: X has 9 features, but StandardScaler is expecting 4 features as input.

>>> TEST [DLINEAR]: VNM
  [OK] Scaler loaded: price_dim=4, sentiment_dim=4


...

 THỐNG KÊ FILE GỐC:
   Price     : 20 dòng
   Sentiment : 718 dòng
  [INFO] Dữ liệu TEST nhỏ (20 dòng): điều chỉnh seq_len từ 30 → 19
    [WARNING] No sentiment columns found!
    [INFO] NODE: Chỉ dùng Price (dummy sentiment: 4 cols)
  Lỗi tại AMAZON: X has 1 features, but StandardScaler is expecting 4 features as input.

[SKIP] APPLE: Không tìm thấy sentiment file trong TEST.

>>> TEST [NODE]: GOOGLE
  [OK] Scaler loaded: price_dim=4, sentiment_dim=4
 THỐNG KÊ FILE GỐC:
   Price     : 20 dòng
   Sentiment : 262 dòng
  [INFO] Dữ liệu TEST nhỏ (20 dòng): điều chỉnh seq_len từ 30 → 19
    [INFO] NODE: Chỉ dùng Price (dummy sentiment: 4 cols)
  Lỗi tại GOOGLE: X has 9 features, but StandardScaler is expecting 4 features as input.

>>> TEST [NODE]: META
  [OK] Scaler loaded: price_dim=4, sentiment_dim=4
 THỐNG KÊ FILE GỐC:
   Price     : 20 dòng
   Sentiment : 143 dòng
  [INFO] Dữ liệu TEST nhỏ (20 dòng): điều chỉnh seq_len từ 30 → 19
    [INFO] NODE: Chỉ dùng Price (dummy sentiment: 4 col

Traceback (most recent call last):
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_624\3001265464.py", line 135, in <module>
    X_p_test, X_s_test, y_test = processor.prepare_sequences(
                                 ~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        price_f_test, sent_f_test, targets_test,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        ticker_core, actual_seq_len, fit=False  # ← QUAN TRỌNG: fit=False
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_624\971465480.py", line 95, in prepare_sequences
    s_scaled = self.sentiment_scaler.transform(s_data)
  File "d:\NghienCuu\NCT3\.venv\Lib\site-packages\sklearn\utils\_set_output.py", line 316, in wrapped
    data_to_wrap = f(self, X, *args, **kwargs)
  File "d:\NghienCuu\NCT3\.venv\Lib\site-packages\sklearn\preprocessing\_data.py", line 1094, in transform
    X = validate_data(
        self,
    ...<6 lines>...
        ensure_all_f


>>> TEST [NODE]: VNM
  [OK] Scaler loaded: price_dim=4, sentiment_dim=4
 THỐNG KÊ FILE GỐC:
   Price     : 20 dòng
   Sentiment : 1166 dòng
  [INFO] Dữ liệu TEST nhỏ (20 dòng): điều chỉnh seq_len từ 30 → 19
    [WARNING] No sentiment columns found!
    [INFO] NODE: Chỉ dùng Price (dummy sentiment: 4 cols)
  Lỗi tại VNM: X has 1 features, but StandardScaler is expecting 4 features as input.

 TESTING: Model Type = HYBRID

>>> TEST [HYBRID]: ALIBABA
  [OK] Scaler loaded: price_dim=4, sentiment_dim=1
 THỐNG KÊ FILE GỐC:
   Price     : 20 dòng
   Sentiment : 164 dòng
  [INFO] Dữ liệu TEST nhỏ (20 dòng): điều chỉnh seq_len từ 30 → 19
    [INFO] Placeholder detected in TRAIN scaler → allowing auto-discovery in TEST
    [WARNING] No sentiment columns found!
  [SKIP] Model file not found: ..\..\LOGS\DLINEAR+NODE\ALIBABA_hybrid.pt

>>> TEST [HYBRID]: AMAZON
  [OK] Scaler loaded: price_dim=4, sentiment_dim=1


Traceback (most recent call last):
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_624\3001265464.py", line 135, in <module>
    X_p_test, X_s_test, y_test = processor.prepare_sequences(
                                 ~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        price_f_test, sent_f_test, targets_test,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        ticker_core, actual_seq_len, fit=False  # ← QUAN TRỌNG: fit=False
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\vitra\AppData\Local\Temp\ipykernel_624\971465480.py", line 95, in prepare_sequences
    s_scaled = self.sentiment_scaler.transform(s_data)
  File "d:\NghienCuu\NCT3\.venv\Lib\site-packages\sklearn\utils\_set_output.py", line 316, in wrapped
    data_to_wrap = f(self, X, *args, **kwargs)
  File "d:\NghienCuu\NCT3\.venv\Lib\site-packages\sklearn\preprocessing\_data.py", line 1094, in transform
    X = validate_data(
        self,
    ...<6 lines>...
        ensure_all_f

 THỐNG KÊ FILE GỐC:
   Price     : 20 dòng
   Sentiment : 718 dòng
  [INFO] Dữ liệu TEST nhỏ (20 dòng): điều chỉnh seq_len từ 30 → 19
    [INFO] Placeholder detected in TRAIN scaler → allowing auto-discovery in TEST
    [WARNING] No sentiment columns found!
  [SKIP] Model file not found: ..\..\LOGS\DLINEAR+NODE\AMAZON_hybrid.pt

[SKIP] APPLE: Không tìm thấy sentiment file trong TEST.

>>> TEST [HYBRID]: GOOGLE
  [OK] Scaler loaded: price_dim=4, sentiment_dim=1
 THỐNG KÊ FILE GỐC:
   Price     : 20 dòng
   Sentiment : 262 dòng
  [INFO] Dữ liệu TEST nhỏ (20 dòng): điều chỉnh seq_len từ 30 → 19
    [INFO] Placeholder detected in TRAIN scaler → allowing auto-discovery in TEST
    [INFO] Placeholder→Real sentiment cols mismatch detected
           Refitted sentiment_scaler for 9 features
  [SKIP] Model file not found: ..\..\LOGS\DLINEAR+NODE\GOOGLE_hybrid.pt

>>> TEST [HYBRID]: META
  [OK] Scaler loaded: price_dim=4, sentiment_dim=1
 THỐNG KÊ FILE GỐC:
   Price     : 20 dòng
   Sentiment : 

In [31]:
# =============================================================================
# CELL 7: HIỂN THỊ KẾT QUẢ TEST & SO SÁNH 3 MODEL TYPE
# =============================================================================

print("\n" + "="*100)
print(" BẢNG TỔNG HỢP ĐÁNH GIÁ MODEL TRÊN TEST SET (SO SÁNH 3 MODEL TYPE)")
print("="*100)

# Đọc và hiển thị bảng test_evaluation_summary.csv
test_summary_path = os.path.join(LOG_DIR, 'test_evaluation_summary.csv')
if os.path.exists(test_summary_path):
    test_summary = pd.read_csv(test_summary_path)
    
    # Check if Model_Type column exists (for backward compatibility with old CSV files)
    if 'Model_Type' not in test_summary.columns:
        print("[WARNING] 'Model_Type' column not found in test_evaluation_summary.csv")
        print("[INFO] This usually means the CSV was created by a previous run before Model_Type was added.")
        print("[FIX] Please re-run CELL 6 (TEST) to regenerate the summary with all 3 model types.\n")
        # Show available columns
        print(f"Available columns: {list(test_summary.columns)}")
        available_cols = [c for c in test_summary.columns if c in ['Ticker', 'Mode', 'DA (%)', 'MAPE (%)', 'RMSE', 'MAE', 'R2']]
    else:
        available_cols = ['Ticker', 'Model_Type', 'Mode', 'DA (%)', 'MAPE (%)', 'RMSE', 'MAE', 'R2']
    
    # Filter to only available columns
    display_cols = [c for c in available_cols if c in test_summary.columns]
    
    print("\n")
    display(test_summary[display_cols].round(4))
    print("\n" + "="*100)
    print(f"✓ Chi tiết đầy đủ: {test_summary_path}")
    
    # Tạo bảng so sánh tổng hợp (pivot table) - chỉ nếu có Model_Type column
    if 'Model_Type' in test_summary.columns:
        print("\n" + "="*100)
        print(" BẢNG SO SÁNH TỔNG HỢP (PIVOT TABLE)")
        print("="*100)
        
        for metric in ['DA (%)', 'MAPE (%)', 'RMSE', 'MAE', 'R2']:
            if metric in test_summary.columns:
                pivot_table = test_summary.pivot_table(
                    index='Ticker', 
                    columns='Model_Type', 
                    values=metric,
                    aggfunc='first'
                )
                print(f"\n{metric}:")
                display(pivot_table.round(4))
        
        # Lưu bảng so sánh
        comparison_path = os.path.join(LOG_DIR, 'model_comparison_summary.csv')
        test_summary.to_csv(comparison_path, index=False, encoding='utf-8-sig')
        print(f"\nĐã lưu model_comparison_summary.csv")
    else:
        print("\n[SKIP] Không thể tạo pivot table vì thiếu 'Model_Type' column.")
        print("       Hãy re-run CELL 6 trước.")
else:
    print(f"[WARNING] Không tìm thấy {test_summary_path}")

# Hiển thị dự báo 5 ngày tương lai cho từng stock và model type
print("\n" + "="*100)
print(" DỰ BÁO 5 NGÀY TƯƠNG LAI (CẢ 3 MODEL TYPE)")
print("="*100)

# Tìm tất cả file predict
predict_files = [f for f in os.listdir(LOG_DIR) if f.endswith('_predict.csv')]
predict_files.sort()

# Khởi tạo sẵn để tránh lỗi undefined
from collections import defaultdict
predict_by_ticker = defaultdict(list)

if len(predict_files) == 0:
    print("[INFO] Không tìm thấy file dự báo. Hãy re-run CELL 6 trước.")
else:
    # Phân nhóm theo ticker
    for predict_file in predict_files:
        parts = predict_file.replace('_predict.csv', '').split('_')
        if len(parts) >= 2:
            ticker = parts[0]
            model_type = '_'.join(parts[1:])
            predict_by_ticker[ticker].append((model_type, predict_file))

    # Hiển thị theo ticker
    for ticker in sorted(predict_by_ticker.keys()):
        print(f"\n{'═'*100}")
        print(f" {ticker} — BẢNG SO SÁNH 3 MODEL TYPE")
        print(f"{'═'*100}")
        
        for model_type, predict_file in sorted(predict_by_ticker[ticker]):
            predict_path = os.path.join(LOG_DIR, predict_file)
            predict_df = pd.read_csv(predict_path)
            future_df = predict_df[predict_df['Type'] == 'Future'][['Date', 'Open', 'High', 'Low', 'Close']]
            
            if len(future_df) > 0:
                print(f"\n{model_type.upper()}:")
                print(future_df.to_string(index=False))
                
                # Thống kê dự báo
                actual_df = predict_df[predict_df['Type'] == 'Predicted']
                if len(actual_df) > 0:
                    last_actual = actual_df['Actual'].iloc[-1]
                    pred_close = future_df['Close'].iloc[0]
                    change_pct = ((pred_close - last_actual) / last_actual) * 100
                    print(f"  Giá cuối (ngày cuối TEST): {last_actual:.2f}")
                    print(f"  Dự báo ngày đầu tiên: {pred_close:.2f} ({change_pct:+.2f}%)")

print("\n" + "="*100)
print(" BIỂU ĐỒ SO SÁNH HIỆU SUẤT 3 MÔ HÌNH")
print("="*100)

if os.path.exists(test_summary_path) and 'Model_Type' in test_summary.columns and len(test_summary) > 0:
    try:
        # Tạo biểu đồ so sánh metrics
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle('So Sánh Hiệu Suất: DLinear vs NODE vs Hybrid', fontsize=16, fontweight='bold')
        plt.subplots_adjust(hspace=0.3, wspace=0.3)
        
        metrics = ['DA (%)', 'MAPE (%)', 'RMSE', 'MAE', 'R2']
        model_types = ['dlinear', 'node', 'hybrid']
        colors = {'dlinear': '#FF6B6B', 'node': '#4ECDC4', 'hybrid': '#45B7D1'}
        
        # 1. DA (%) - Directional Accuracy
        ax = axes[0, 0]
        for ticker in test_summary['Ticker'].unique():
            ticker_data = test_summary[test_summary['Ticker'] == ticker]
            values = []
            for model_type in model_types:
                val = ticker_data[ticker_data['Model_Type'] == model_type]['DA (%)'].values
                values.append(val[0] if len(val) > 0 else 0)
            ax.bar([f'{ticker}\n{mt}' for mt in model_types], values, 
                   color=[colors[mt] for mt in model_types], alpha=0.8, width=0.6)
        ax.set_ylabel('DA (%)', fontsize=11, fontweight='bold')
        ax.set_title('Directional Accuracy (%)\n(Cao hơn = Tốt hơn)', fontsize=11)
        ax.grid(axis='y', alpha=0.3)
        ax.axhline(y=50, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Random (50%)')
        ax.legend()
        
        # 2. MAPE (%) - Mean Absolute Percentage Error
        ax = axes[0, 1]
        for ticker in test_summary['Ticker'].unique():
            ticker_data = test_summary[test_summary['Ticker'] == ticker]
            values = []
            for model_type in model_types:
                val = ticker_data[ticker_data['Model_Type'] == model_type]['MAPE (%)'].values
                values.append(val[0] if len(val) > 0 else 0)
            ax.bar([f'{ticker}\n{mt}' for mt in model_types], values,
                   color=[colors[mt] for mt in model_types], alpha=0.8, width=0.6)
        ax.set_ylabel('MAPE (%)', fontsize=11, fontweight='bold')
        ax.set_title('Mean Absolute Percentage Error (%)\n(Thấp hơn = Tốt hơn)', fontsize=11)
        ax.grid(axis='y', alpha=0.3)
        
        # 3. RMSE - Root Mean Squared Error
        ax = axes[0, 2]
        for ticker in test_summary['Ticker'].unique():
            ticker_data = test_summary[test_summary['Ticker'] == ticker]
            values = []
            for model_type in model_types:
                val = ticker_data[ticker_data['Model_Type'] == model_type]['RMSE'].values
                values.append(val[0] if len(val) > 0 else 0)
            ax.bar([f'{ticker}\n{mt}' for mt in model_types], values,
                   color=[colors[mt] for mt in model_types], alpha=0.8, width=0.6)
        ax.set_ylabel('RMSE', fontsize=11, fontweight='bold')
        ax.set_title('Root Mean Squared Error\n(Thấp hơn = Tốt hơn)', fontsize=11)
        ax.grid(axis='y', alpha=0.3)
        
        # 4. MAE - Mean Absolute Error
        ax = axes[1, 0]
        for ticker in test_summary['Ticker'].unique():
            ticker_data = test_summary[test_summary['Ticker'] == ticker]
            values = []
            for model_type in model_types:
                val = ticker_data[ticker_data['Model_Type'] == model_type]['MAE'].values
                values.append(val[0] if len(val) > 0 else 0)
            ax.bar([f'{ticker}\n{mt}' for mt in model_types], values,
                   color=[colors[mt] for mt in model_types], alpha=0.8, width=0.6)
        ax.set_ylabel('MAE', fontsize=11, fontweight='bold')
        ax.set_title('Mean Absolute Error\n(Thấp hơn = Tốt hơn)', fontsize=11)
        ax.grid(axis='y', alpha=0.3)
        
        # 5. R2 - R-squared Score
        ax = axes[1, 1]
        for ticker in test_summary['Ticker'].unique():
            ticker_data = test_summary[test_summary['Ticker'] == ticker]
            values = []
            for model_type in model_types:
                val = ticker_data[ticker_data['Model_Type'] == model_type]['R2'].values
                values.append(val[0] if len(val) > 0 else 0)
            ax.bar([f'{ticker}\n{mt}' for mt in model_types], values,
                   color=[colors[mt] for mt in model_types], alpha=0.8, width=0.6)
        ax.set_ylabel('R2', fontsize=11, fontweight='bold')
        ax.set_title('R-squared Score\n(Cao hơn = Tốt hơn)', fontsize=11)
        ax.grid(axis='y', alpha=0.3)
        ax.axhline(y=0.8, color='green', linestyle='--', linewidth=1, alpha=0.5, label='Good (0.8)')
        ax.legend()
        
        # 6. Legend / Model Info
        ax = axes[1, 2]
        ax.axis('off')
        legend_text = (
            "CHỈ SỐ ĐÁNH GIÁ\n"
            "━" * 40 + "\n\n"
            "🔴 DLinear: Chỉ dùng Price\n"
            "🟢 NODE: Chỉ dùng Price\n"
            "🔵 Hybrid: Price + Sentiment\n\n"
            "THANG ĐO:\n"
            "━" * 40 + "\n"
            "✓ DA (%): Cao hơn tốt hơn (>50%)\n"
            "✓ MAPE (%): Thấp hơn tốt hơn\n"
            "✓ RMSE: Thấp hơn tốt hơn\n"
            "✓ MAE: Thấp hơn tốt hơn\n"
            "✓ R2: Cao hơn tốt hơn (0-1)\n\n"
            "KẾT LUẬN:\n"
            "━" * 40 + "\n"
            "Biểu đồ này so sánh 3 mô hình\n"
            "trên cùng dataset TEST"
        )
        ax.text(0.1, 0.5, legend_text, fontsize=10, verticalalignment='center',
                fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
        
        chart_path = os.path.join(CHART_DIR, 'metrics_comparison_3models.png')
        plt.savefig(chart_path, bbox_inches='tight', dpi=150)
        plt.show()
        print(f"\n✓ Biểu đồ lưu tại: {chart_path}")
    except Exception as e:
        print(f"[WARNING] Lỗi khi vẽ biểu đồ: {e}")

print("\n" + "="*100)
print(" PHÂN TÍCH SO SÁNH: MÔ HÌNH ĐƠN vs HYBRID")
print("="*100)

if os.path.exists(test_summary_path) and 'Model_Type' in test_summary.columns:
    # Tạo bảng so sánh chi tiết: Standalone (DLinear, NODE) vs Hybrid
    comparison_data = []
    
    # Nhóm theo ticker
    for ticker in test_summary['Ticker'].unique():
        ticker_data = test_summary[test_summary['Ticker'] == ticker]
        
        hybrid_row = ticker_data[ticker_data['Model_Type'] == 'hybrid']
        dlinear_row = ticker_data[ticker_data['Model_Type'] == 'dlinear']
        node_row = ticker_data[ticker_data['Model_Type'] == 'node']
        
        if len(hybrid_row) == 0:
            continue
        
        # Lấy giá trị của hybrid
        h_da = hybrid_row['DA (%)'].values[0] if len(hybrid_row) > 0 else None
        h_mape = hybrid_row['MAPE (%)'].values[0] if len(hybrid_row) > 0 else None
        h_rmse = hybrid_row['RMSE'].values[0] if len(hybrid_row) > 0 else None
        h_mae = hybrid_row['MAE'].values[0] if len(hybrid_row) > 0 else None
        h_r2 = hybrid_row['R2'].values[0] if len(hybrid_row) > 0 else None
        
        # DLinear vs Hybrid
        if len(dlinear_row) > 0:
            d_da = dlinear_row['DA (%)'].values[0]
            d_mape = dlinear_row['MAPE (%)'].values[0]
            d_rmse = dlinear_row['RMSE'].values[0]
            d_mae = dlinear_row['MAE'].values[0]
            d_r2 = dlinear_row['R2'].values[0]
            
            comparison_data.append({
                'Ticker': ticker,
                'So Sánh': 'DLinear vs Hybrid',
                'DA(%) - DLinear': d_da,
                'DA(%) - Hybrid': h_da,
                'DA(%) - Diff': d_da - h_da,
                'MAPE(%) - DLinear': d_mape,
                'MAPE(%) - Hybrid': h_mape,
                'MAPE(%) - Diff': d_mape - h_mape,
                'RMSE - DLinear': d_rmse,
                'RMSE - Hybrid': h_rmse,
                'RMSE - Diff': d_rmse - h_rmse,
                'MAE - DLinear': d_mae,
                'MAE - Hybrid': h_mae,
                'MAE - Diff': d_mae - h_mae,
                'R2 - DLinear': d_r2,
                'R2 - Hybrid': h_r2,
                'R2 - Diff': d_r2 - h_r2,
            })
        
        # NODE vs Hybrid
        if len(node_row) > 0:
            n_da = node_row['DA (%)'].values[0]
            n_mape = node_row['MAPE (%)'].values[0]
            n_rmse = node_row['RMSE'].values[0]
            n_mae = node_row['MAE'].values[0]
            n_r2 = node_row['R2'].values[0]
            
            comparison_data.append({
                'Ticker': ticker,
                'So Sánh': 'NODE vs Hybrid',
                'DA(%) - NODE': n_da,
                'DA(%) - Hybrid': h_da,
                'DA(%) - Diff': n_da - h_da,
                'MAPE(%) - NODE': n_mape,
                'MAPE(%) - Hybrid': h_mape,
                'MAPE(%) - Diff': n_mape - h_mape,
                'RMSE - NODE': n_rmse,
                'RMSE - Hybrid': h_rmse,
                'RMSE - Diff': n_rmse - h_rmse,
                'MAE - NODE': n_mae,
                'MAE - Hybrid': h_mae,
                'MAE - Diff': n_mae - h_mae,
                'R2 - NODE': n_r2,
                'R2 - Hybrid': h_r2,
                'R2 - Diff': n_r2 - h_r2,
            })
    
    if comparison_data:
        comparison_df = pd.DataFrame(comparison_data)
        
        # Hiển thị DLinear vs Hybrid
        print("\n" + "─"*100)
        print(" DLinear vs Hybrid")
        print("─"*100)
        dlinear_vs_hybrid = comparison_df[comparison_df['So Sánh'] == 'DLinear vs Hybrid']
        if len(dlinear_vs_hybrid) > 0:
            dlinear_display = dlinear_vs_hybrid[['Ticker', 'DA(%) - DLinear', 'DA(%) - Hybrid', 'DA(%) - Diff',
                                                   'MAPE(%) - DLinear', 'MAPE(%) - Hybrid', 'MAPE(%) - Diff',
                                                   'RMSE - DLinear', 'RMSE - Hybrid', 'RMSE - Diff',
                                                   'MAE - DLinear', 'MAE - Hybrid', 'MAE - Diff',
                                                   'R2 - DLinear', 'R2 - Hybrid', 'R2 - Diff']]
            display(dlinear_display.round(4))
        
        # Hiển thị NODE vs Hybrid
        print("\n" + "─"*100)
        print(" NODE vs Hybrid")
        print("─"*100)
        node_vs_hybrid = comparison_df[comparison_df['So Sánh'] == 'NODE vs Hybrid']
        if len(node_vs_hybrid) > 0:
            node_display = node_vs_hybrid[['Ticker', 'DA(%) - NODE', 'DA(%) - Hybrid', 'DA(%) - Diff',
                                            'MAPE(%) - NODE', 'MAPE(%) - Hybrid', 'MAPE(%) - Diff',
                                            'RMSE - NODE', 'RMSE - Hybrid', 'RMSE - Diff',
                                            'MAE - NODE', 'MAE - Hybrid', 'MAE - Diff',
                                            'R2 - NODE', 'R2 - Hybrid', 'R2 - Diff']]
            display(node_display.round(4))
        
        # Tính toán tổng kết
        print("\n" + "─"*100)
        print(" TỔNG KẾT HIỆU SUẤT")
        print("─"*100)
        
        # DLinear so với Hybrid
        if len(dlinear_vs_hybrid) > 0:
            print("\n✓ DLinear so với Hybrid:")
            da_diff_mean = dlinear_vs_hybrid['DA(%) - Diff'].mean()
            mape_diff_mean = dlinear_vs_hybrid['MAPE(%) - Diff'].mean()
            rmse_diff_mean = dlinear_vs_hybrid['RMSE - Diff'].mean()
            mae_diff_mean = dlinear_vs_hybrid['MAE - Diff'].mean()
            r2_diff_mean = dlinear_vs_hybrid['R2 - Diff'].mean()
            
            print(f"  DA(%):    {da_diff_mean:+.2f}% (Cao hơn = tốt hơn)")
            print(f"  MAPE(%):  {mape_diff_mean:+.2f}% (Thấp hơn = tốt hơn)")
            print(f"  RMSE:     {rmse_diff_mean:+.4f} (Thấp hơn = tốt hơn)")
            print(f"  MAE:      {mae_diff_mean:+.4f} (Thấp hơn = tốt hơn)")
            print(f"  R2:       {r2_diff_mean:+.4f} (Cao hơn = tốt hơn)")
            
            # Kết luận
            if da_diff_mean > 1 and mape_diff_mean > 1 and r2_diff_mean > 0.01:
                print("  → DLinear TỐT HƠN Hybrid")
            elif da_diff_mean < -1 and mape_diff_mean < -1:
                print("  → DLinear KÉM HƠN Hybrid")
            else:
                print("  → DLinear TƯƠNG ĐƯƠNG với Hybrid")
        
        # NODE so với Hybrid
        if len(node_vs_hybrid) > 0:
            print("\n✓ NODE so với Hybrid:")
            da_diff_mean = node_vs_hybrid['DA(%) - Diff'].mean()
            mape_diff_mean = node_vs_hybrid['MAPE(%) - Diff'].mean()
            rmse_diff_mean = node_vs_hybrid['RMSE - Diff'].mean()
            mae_diff_mean = node_vs_hybrid['MAE - Diff'].mean()
            r2_diff_mean = node_vs_hybrid['R2 - Diff'].mean()
            
            print(f"  DA(%):    {da_diff_mean:+.2f}% (Cao hơn = tốt hơn)")
            print(f"  MAPE(%):  {mape_diff_mean:+.2f}% (Thấp hơn = tốt hơn)")
            print(f"  RMSE:     {rmse_diff_mean:+.4f} (Thấp hơn = tốt hơn)")
            print(f"  MAE:      {mae_diff_mean:+.4f} (Thấp hơn = tốt hơn)")
            print(f"  R2:       {r2_diff_mean:+.4f} (Cao hơn = tốt hơn)")
            
            # Kết luận
            if da_diff_mean > 1 and mape_diff_mean > 1 and r2_diff_mean > 0.01:
                print("  → NODE TỐT HƠN Hybrid")
            elif da_diff_mean < -1 and mape_diff_mean < -1:
                print("  → NODE KÉM HƠN Hybrid")
            else:
                print("  → NODE TƯƠNG ĐƯƠNG với Hybrid")
        
        # Xuất file so sánh
        comparison_output_path = os.path.join(LOG_DIR, 'standalone_vs_hybrid_comparison.csv')
        comparison_df.to_csv(comparison_output_path, index=False, encoding='utf-8-sig')
        print(f"\n✓ Đã lưu bảng so sánh chi tiết: {comparison_output_path}")
        
        # Vẽ biểu đồ so sánh DLinear vs Hybrid và NODE vs Hybrid
        try:
            fig, axes = plt.subplots(1, 2, figsize=(16, 6))
            fig.suptitle('So Sánh Standalone Models vs Hybrid', fontsize=16, fontweight='bold')
            
            # DLinear vs Hybrid
            dlinear_data = comparison_df[comparison_df['So Sánh'] == 'DLinear vs Hybrid']
            if len(dlinear_data) > 0:
                ax = axes[0]
                x_pos = np.arange(len(dlinear_data))
                width = 0.35
                
                ax.bar(x_pos - width/2, dlinear_data['MAPE(%) - DLinear'], width, 
                       label='DLinear', color='#FF6B6B', alpha=0.8)
                ax.bar(x_pos + width/2, dlinear_data['MAPE(%) - Hybrid'], width,
                       label='Hybrid', color='#45B7D1', alpha=0.8)
                
                ax.set_xlabel('Stock Ticker', fontsize=11, fontweight='bold')
                ax.set_ylabel('MAPE (%)', fontsize=11, fontweight='bold')
                ax.set_title('DLinear vs Hybrid - MAPE\n(Thấp hơn = Tốt hơn)', fontsize=12, fontweight='bold')
                ax.set_xticks(x_pos)
                ax.set_xticklabels(dlinear_data['Ticker'])
                ax.legend()
                ax.grid(axis='y', alpha=0.3)
            
            # NODE vs Hybrid
            node_data = comparison_df[comparison_df['So Sánh'] == 'NODE vs Hybrid']
            if len(node_data) > 0:
                ax = axes[1]
                x_pos = np.arange(len(node_data))
                width = 0.35
                
                ax.bar(x_pos - width/2, node_data['MAPE(%) - NODE'], width,
                       label='NODE', color='#4ECDC4', alpha=0.8)
                ax.bar(x_pos + width/2, node_data['MAPE(%) - Hybrid'], width,
                       label='Hybrid', color='#45B7D1', alpha=0.8)
                
                ax.set_xlabel('Stock Ticker', fontsize=11, fontweight='bold')
                ax.set_ylabel('MAPE (%)', fontsize=11, fontweight='bold')
                ax.set_title('NODE vs Hybrid - MAPE\n(Thấp hơn = Tốt hơn)', fontsize=12, fontweight='bold')
                ax.set_xticks(x_pos)
                ax.set_xticklabels(node_data['Ticker'])
                ax.legend()
                ax.grid(axis='y', alpha=0.3)
            
            chart_path = os.path.join(CHART_DIR, 'standalone_vs_hybrid_comparison.png')
            plt.savefig(chart_path, bbox_inches='tight', dpi=150)
            plt.show()
            print(f"✓ Biểu đồ so sánh lưu tại: {chart_path}")
        except Exception as e:
            print(f"[WARNING] Lỗi khi vẽ biểu đồ so sánh: {e}")
    else:
        print("[WARNING] Không đủ dữ liệu để so sánh (cần cả standalone và hybrid models)")
else:
    print("[SKIP] Chưa có dữ liệu test hoặc Model_Type column chưa được khởi tạo.")
    print("       Hãy re-run CELL 6 trước.")

print("\n" + "="*100)
print(" THÔNG TIN FILE ĐƯỢC LƯU")
print("="*100)
print(f"\n📊 Bảng so sánh:")
print(f"   {os.path.join(LOG_DIR, 'test_evaluation_summary.csv')}")
print(f"   {os.path.join(LOG_DIR, 'model_comparison_summary.csv')}")
print(f"   {os.path.join(LOG_DIR, 'standalone_vs_hybrid_comparison.csv')}")
print(f"\n📈 File dự báo từng stock × model:")
if 'predict_by_ticker' in locals():
    for ticker in sorted(predict_by_ticker.keys()):
        for model_type, predict_file in sorted(predict_by_ticker[ticker]):
            predict_path = os.path.join(LOG_DIR, predict_file)
            print(f"   {predict_path}")
print(f"\n📷 Biểu đồ:")
print(f"   (Lưu trong {os.path.join(CHART_DIR)})")


 BẢNG TỔNG HỢP ĐÁNH GIÁ MODEL TRÊN TEST SET (SO SÁNH 3 MODEL TYPE)
[WARNING] 'Model_Type' column not found in test_evaluation_summary.csv
[INFO] This usually means the CSV was created by a previous run before Model_Type was added.
[FIX] Please re-run CELL 6 (TEST) to regenerate the summary with all 3 model types.

Available columns: ['Ticker', 'Mode', 'MSE', 'RMSE', 'MAE', 'MAPE (%)', 'R2', 'DA (%)', 'Actual Last', 'Pred Last']




,Ticker,Mode,RMSE,MAE,MAPE (%),R2,DA (%)
0,ALIBABA,TEST,4.5405,3.2449,2.5100,0.9682,53.8813
1,AMAZON,TEST,5.0815,3.5828,1.6950,0.9167,52.5114
2,GOOGLE,TEST,4.3785,3.3270,1.7314,0.9862,51.1416
3,META,TEST,28.0434,24.2611,3.6053,0.8482,46.5753
4,VNM,TEST,0.3307,0.2456,1.7432,0.9848,52.5114



✓ Chi tiết đầy đủ: ..\..\LOGS\DLINEAR+NODE\test_evaluation_summary.csv

[SKIP] Không thể tạo pivot table vì thiếu 'Model_Type' column.
       Hãy re-run CELL 6 trước.

 DỰ BÁO 5 NGÀY TƯƠNG LAI (CẢ 3 MODEL TYPE)

 BIỂU ĐỒ SO SÁNH HIỆU SUẤT 3 MÔ HÌNH

 PHÂN TÍCH SO SÁNH: MÔ HÌNH ĐƠN vs HYBRID
[SKIP] Chưa có dữ liệu test hoặc Model_Type column chưa được khởi tạo.
       Hãy re-run CELL 6 trước.

 THÔNG TIN FILE ĐƯỢC LƯU

📊 Bảng so sánh:
   ..\..\LOGS\DLINEAR+NODE\test_evaluation_summary.csv
   ..\..\LOGS\DLINEAR+NODE\model_comparison_summary.csv
   ..\..\LOGS\DLINEAR+NODE\standalone_vs_hybrid_comparison.csv

📈 File dự báo từng stock × model:

📷 Biểu đồ:
   (Lưu trong ..\..\CHART\DLINEAR+NODE)


In [32]:
# =============================================================================
# DIAGNOSTIC CELL: Check TEST Files Content
# =============================================================================

print("="*100)
print(" KIỂM TRA NỘI DUNG FILE TEST")
print("="*100)

# Check a few price files
test_price_files_list = [f for f in os.listdir(TEST_PRICE_DIR) if f.endswith('.csv')]
print(f"\nTEST PRICE FILES ({len(test_price_files_list)} files):")
for fname in test_price_files_list[:3]:  # Check first 3 files
    fpath = os.path.join(TEST_PRICE_DIR, fname)
    df_check = pd.read_csv(fpath, encoding='utf-8-sig', nrows=5)
    print(f"\n  📄 {fname}:")
    print(f"     Shape: {df_check.shape}")
    print(f"     Columns: {list(df_check.columns)}")
    print(f"     First row:\n{df_check.iloc[0].to_string()}")

# Check a few sentiment files  
test_sent_files_list = [f for f in os.listdir(TEST_SENTIMENT_DIR) if f.endswith('.csv')]
print(f"\n\nTEST SENTIMENT FILES ({len(test_sent_files_list)} files):")
for fname in test_sent_files_list[:3]:  # Check first 3 files
    fpath = os.path.join(TEST_SENTIMENT_DIR, fname)
    df_check = pd.read_csv(fpath, encoding='utf-8-sig', nrows=5)
    print(f"\n  📄 {fname}:")
    print(f"     Shape: {df_check.shape}")
    print(f"     Columns: {list(df_check.columns)}")
    print(f"     First row:\n{df_check.iloc[0].to_string()}")


 KIỂM TRA NỘI DUNG FILE TEST

TEST PRICE FILES (6 files):

  📄 ALIBABA.csv:
     Shape: (5, 6)
     Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
     First row:
Date      11/15/2024
Open            90.2
High            90.7
Low            87.23
Close          88.59
Volume      31018300

  📄 AMAZON.csv:
     Shape: (5, 6)
     Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
     First row:
Date      11/15/2024
Open          206.76
High          207.34
Low           199.61
Close         202.61
Volume      86591100

  📄 APPLE.csv:
     Shape: (5, 6)
     Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
     First row:
Date      11/15/2024
Open           226.4
High          226.92
Low           224.27
Close          225.0
Volume      47923700


TEST SENTIMENT FILES (8 files):

  📄 ALIBABA.csv:
     Shape: (5, 3)
     Columns: ['Date', 'Header', 'Content']
     First row:
Date                                              2025-01-12
Header     Morgan Sta